# In this notebook, I will turn search space dictionaries into pickle files to be read by my job array

In [12]:
import pickle
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from skopt.space import Categorical, Real, Integer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import itertools
import numpy as np
import pandas as pd
from pyhere import here

In [13]:
class RatioGenerator(BaseEstimator, TransformerMixin):
    '''
    A custom transformer that generates new features by taking the ratios of all combinations of specified columns.
    For use with the flux columns
    '''
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        # add a dummy attribute so sklearn knows this transformer is fitted
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Input X must be a pandas DataFrame")
        
        # Create a copy to avoid SettingWithCopy warnings or mutating the original
        X_out = X.copy()
        
        for top, bottom in itertools.combinations(self.cols, 2):
            new_col_name = f"{top}_over_{bottom}"
            X_out[new_col_name] = X_out[top] / (X_out[bottom] + 1e-8 ) #add epsilon to reduce division by 0 errors
            
        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            raise ValueError("input_features must be provided")

        input_features = list(input_features)

        # Validate columns exist
        missing = set(self.cols) - set(input_features)
        if missing:
            raise ValueError(f"Missing columns in input_features: {missing}")

        # Generate ratio feature names
        ratio_features = [
            f"{top}_over_{bottom}"
            for top, bottom in itertools.combinations(self.cols, 2)
        ]

        # IMPORTANT: include ALL original input features
        return np.array(input_features + ratio_features, dtype=object)

flux_cols = ['F8', 'F12', 'F24', 'F70', 'F160', 'F250', 'F350', 'F500', 'F870', 'F1100']

In [14]:
catboost =  {
    "pipe": Pipeline([
        ('impute', SimpleImputer()),
        ('ratio', RatioGenerator(cols=flux_cols)),
        ('scale', RobustScaler()),
        ('model', CatBoostRegressor(random_state=2026, verbose=0, thread_count=-1, loss_function='RMSE'))
    ]),
    "space": {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer(), 'passthrough']),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__iterations": Integer(100, 3000),
        "model__learning_rate": Real(1e-4, 0.5, prior="log-uniform"),
        "model__depth": Integer(3, 12),
        "model__l2_leaf_reg": Real(1e-3, 100.0, prior="log-uniform"),
        "model__random_strength": Real(1e-9, 10.0, prior="log-uniform"),
        "model__bagging_temperature": Real(0.0, 10.0, prior="uniform"),
        "model__border_count": Integer(32, 255),
        "model__min_data_in_leaf": Integer(1, 100),
        "model__colsample_bylevel": Real(0.05, 1.0, prior="uniform"),
        "model__grow_policy": Categorical(["SymmetricTree", "Depthwise", "Lossguide"])
    }       
}
xgboost = {
    "pipe": Pipeline([
        ('impute', SimpleImputer()),
        ('ratio', RatioGenerator(cols=flux_cols)),
        ('scale', RobustScaler()),
        ('model', XGBRegressor(random_state=2026, verbosity=0, n_jobs=-1))
    ]),
    "space": {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer(), 'passthrough']),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__n_estimators": Integer(100, 3000),
        "model__learning_rate": Real(1e-4, 0.5, prior="log-uniform"),
        "model__max_depth": Integer(3, 12),
        "model__min_child_weight": Integer(1, 20),
        "model__subsample": Real(0.5, 1.0, prior="uniform"),
        "model__colsample_bytree": Real(0.5, 1.0, prior="uniform"),
        "model__colsample_bylevel": Real(0.5, 1.0, prior="uniform"),
        "model__reg_alpha": Real(1e-9, 100.0, prior="log-uniform"),
        "model__reg_lambda": Real(1e-9, 100.0, prior="log-uniform"),
        "model__gamma": Real(1e-9, 10.0, prior="log-uniform")
    }
}
random_forest =  {
    "pipe": Pipeline([
        ('impute', SimpleImputer()),
        ('ratio', RatioGenerator(cols=flux_cols)),
        ('scale', RobustScaler()),
        ('model', RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1))
    ]),
    "space": {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer()]),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__n_estimators": Integer(10, 1000),
        "model__max_depth": Integer(3, 30),
        "model__min_samples_split": Integer(2, 20),
        "model__min_samples_leaf": Integer(1, 20),
        "model__max_features": Categorical(["sqrt", "log2", None]), 
        'model__bootstrap': Categorical([True, False])
    }
}
decision_tree=  {
    "pipe": Pipeline([
        ('impute', SimpleImputer()),
        ('ratio', RatioGenerator(cols=flux_cols)),
        ('scale', RobustScaler()),
        ('model', DecisionTreeRegressor(random_state=2026))
    ]),
    "space": {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer()]),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__max_depth": Integer(3, 30),
        "model__min_samples_split": Integer(2, 20),
        "model__min_samples_leaf": Integer(1, 20),
        "model__max_features": Categorical(["sqrt", "log2", None])
    }
}
SVR_space =  {
    "pipe": Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('ratio', RatioGenerator(cols=flux_cols)),
        ('scale', RobustScaler()),
        ('model', SVR())
    ]),
    "space": {
        'scale': Categorical([StandardScaler(), RobustScaler()]),
        'model__kernel': Categorical(['rbf']),
        'model__C': Real(0.1, 100, prior='log-uniform'),
        'model__gamma': Real(1e-4, 1e+1, prior='log-uniform'),
        'model__epsilon': Real(0.01, 1.0, prior='log-uniform')
    }
}


In [15]:
with open('../../pipeline/spaces/catboost_space.pkl', 'wb') as file:
    pickle.dump(catboost, file)

with open('../../pipeline/spaces/xgboost_space.pkl', 'wb') as file:
    pickle.dump(xgboost, file)

with open('../../pipeline/spaces/rf_space.pkl', 'wb') as file:
    pickle.dump(random_forest, file)

with open('../../pipeline/spaces/tree_space.pkl', 'wb') as file:
    pickle.dump(decision_tree, file)

with open('../../pipeline/spaces/svr_space.pkl', 'wb') as file:
    pickle.dump(SVR_space, file)